In [1]:
# Import graph building tools
from langgraph.graph import StateGraph, START, END

# Import Command (for moving between nodes)
# and interrupt (for human input)
from langgraph.types import Command, interrupt

# Import TypedDict to define state structure
from typing import TypedDict

# Import memory saver for checkpointing
from langgraph.checkpoint.memory import MemorySaver


# Create memory object to save graph state
memory = MemorySaver()


# Define graph state structure
class State(TypedDict):
    value: str  # state contains a string value


# First node
def node_a(state: State):

    # Print node name
    print("Node A")

    # Go to node_b and update state
    return Command(
        goto="node_b",  # next node

        update={
            # Add "a" to current value
            "value": state["value"] + "a"
        }
    )


# Second node
def node_b(state: State):

    # Print node name
    print("Node B")

    # Pause execution and ask for user input
    human_response = interrupt(
        "Do you want to go to C or D? Type C/D"
    )

    # Print user response
    print("Human Review Values: ", human_response)

    # If user enters "C"
    if(human_response == "C"):

        # Go to node_c
        return Command(
            goto="node_c",

            update={
                # Add "b" to state value
                "value": state["value"] + "b"
            }
        )

    # If user enters "D"
    elif(human_response == "D"):

        # Go to node_d
        return Command(
            goto="node_d",

            update={
                # Add "b" to state value
                "value": state["value"] + "b"
            }
        )


# Third node
def node_c(state: State):

    # Print node name
    print("Node C")

    # End graph execution
    return Command(
        goto=END,

        update={
            # Add "c" to value
            "value": state["value"] + "c"
        }
    )


# Fourth node
def node_d(state: State):

    # Print node name
    print("Node D")

    # End graph execution
    return Command(
        goto=END,

        update={
            # Add "d" to value
            "value": state["value"] + "d"
        }
    )


# Create graph object using State
graph = StateGraph(State)


# Add all nodes to graph
graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)
graph.add_node("node_c", node_c)
graph.add_node("node_d", node_d)


# Set starting node
graph.set_entry_point("node_a")


# Compile graph with memory checkpointing
app = graph.compile(checkpointer=memory)


# Config with thread id
# Used to remember graph state
config = {
    "configurable": {
        "thread_id": "1"
    }
}


# Initial graph state
initialState = {
    "value": ""
}


# Start graph execution
first_result = app.invoke(
    initialState,  # starting state
    config,        # config settings
    stream_mode="updates"  # return updates
)

# Show result
first_result

c:\Users\HP\Desktop\Jupyter Notebooks\Projects\LangGraph\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Node A
Node B


[{'node_a': {'value': 'a'}},
 {'__interrupt__': (Interrupt(value='Do you want to go to C or D? Type C/D', id='b45add9114acf691ea868b92b741aae4'),)}]

In [2]:
print(app.get_state(config).next)

('node_b',)


In [3]:
second_result = app.invoke(Command(resume="C"), config=config, stream_mode="updates")
second_result

Node B
Human Review Values:  C
Node C


[{'node_b': {'value': 'ab'}}, {'node_c': {'value': 'abc'}}]